# Getting the exposures and returns for each factor

## Set up DB and ensure table exists

In [2]:
import duckdb

new_db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\new_factor_data.duckdb"

conn = duckdb.connect(new_db_path)

conn.execute("SHOW TABLES").fetch_df()

IOException: IO Error: Cannot open file "c:\users\agand\desktop\personal_coding\finance_projects\multifactormodel_lfqa\multi-factor-models\venv\crsp\new_factor_data.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\agand\AppData\Local\Programs\Python\Python313\python.exe (PID 41756)

In [5]:
conn.execute("SELECT * FROM monthly_with_mom").fetch_df()

,cusip,month_start,market_cap_month_end,turnover_month_avg,spread_month_avg,ret_m,mom_12_1
0,00030710,2014-10-01,450257940.0,11.943360,0.004073,0.073459,NaN
1,00030710,2014-11-01,608397660.0,5.657679,0.003756,0.130024,NaN
2,00030710,2014-12-01,651824520.0,4.625977,0.002746,0.030396,NaN
3,00030710,2015-01-01,537987120.0,2.948235,0.001960,-0.079980,NaN
4,00030710,2015-02-01,758705190.0,3.400885,0.002240,0.161022,NaN
...,...,...,...,...,...,...,...
1956553,Y9530810,2023-08-01,430576050.0,2.108568,0.002771,-0.001990,0.070221
1956554,Y9530810,2023-09-01,569319000.0,3.835557,0.002893,0.128827,0.075559
1956555,Y9530810,2023-10-01,532684560.0,2.756272,0.002268,-0.028472,0.035795
1956556,Y9530810,2023-11-01,525976740.0,2.947270,0.002313,-0.005672,0.035053


## Finding top 30% of each factor

In [12]:
conn.execute("""
CREATE OR REPLACE TABLE factor_top30_monthly_returns AS
WITH long AS (
  SELECT
    cusip,
    month_start,
    ret_m,
    factor_name,
    factor_value
  FROM monthly_with_mom
  UNPIVOT(factor_value FOR factor_name IN (
    market_cap_month_end,
    turnover_month_avg,
    spread_month_avg,
    mom_12_1
  ))
  WHERE factor_value IS NOT NULL
    AND ret_m IS NOT NULL
),
ranked AS (
  SELECT
    month_start,
    factor_name,
    ret_m,
    factor_value,
    row_number() OVER (
      PARTITION BY month_start, factor_name
      ORDER BY factor_value DESC
    ) AS rn,
    count(*) OVER (
      PARTITION BY month_start, factor_name
    ) AS n
  FROM long
),
top30 AS (
  SELECT
    month_start,
    factor_name,
    avg(ret_m) AS avg_ret_top30
  FROM ranked
  WHERE rn <= CEIL(0.30 * n)
  GROUP BY month_start, factor_name
)
SELECT *
FROM top30
PIVOT(
  max(avg_ret_top30)
  FOR factor_name IN (
    'market_cap_month_end',
    'turnover_month_avg',
    'spread_month_avg',
    'mom_12_1'
  )
)
ORDER BY month_start;


""")

In [17]:
conn.execute("""
DESCRIBE factor_top30_monthly_returns;
SELECT * FROM factor_top30_monthly_returns LIMIT 13;
""").fetch_df()

,month_start,market_cap_month_end,turnover_month_avg,spread_month_avg,mom_12_1
0,2002-01-01,-0.003480,-0.008239,0.012764,NaN
1,2002-02-01,-0.007157,-0.032389,-0.031926,NaN
2,2002-03-01,0.032602,0.042287,0.024917,NaN
3,2002-04-01,-0.003047,-0.015646,-0.013851,NaN
4,2002-05-01,-0.008325,-0.014750,-0.018461,NaN
5,2002-06-01,-0.027183,-0.043899,-0.056123,NaN
6,2002-07-01,-0.045322,-0.069136,-0.059069,NaN
7,2002-08-01,0.006430,-0.000685,-0.009508,NaN
8,2002-09-01,-0.034730,-0.056079,-0.057441,NaN
9,2002-10-01,0.022628,0.025869,0.004572,NaN


In [23]:
conn.execute("""
             SHOW TABLES
             """).fetch_df()

,name
0,crsp_clean
1,daily_factors_table
2,factor_top30_monthly_returns
3,monthly_base
4,monthly_factors_table
5,monthly_with_mom
6,returns_monthly


## Getting exposures for each cusip

### Joining cusip returns with factor returns and preforming regression

In [40]:
conn.execute("""
CREATE OR REPLACE TABLE cusip_factor_exposures AS
WITH joined AS (
    SELECT
        r.cusip,
        r.month_start,
        r.ret_m,
        f.market_cap_month_end,
        f.turnover_month_avg,
        f.spread_month_avg,
        f.mom_12_1
    FROM returns_monthly r
    JOIN factor_top30_monthly_returns f
        USING (month_start)
    WHERE r.ret_m IS NOT NULL
)
SELECT
    cusip,

    regr_slope(ret_m, market_cap_month_end) AS beta_market_cap,
    regr_slope(ret_m, turnover_month_avg)  AS beta_turnover,
    regr_slope(ret_m, spread_month_avg)    AS beta_spread,
    regr_slope(ret_m, mom_12_1)             AS beta_mom_12_1,

    count(*) AS n_obs
FROM joined
GROUP BY cusip;
""")

In [41]:
conn.execute("SELECT * FROM cusip_factor_exposures ").fetch_df()

,cusip,beta_market_cap,beta_turnover,beta_spread,beta_mom_12_1,n_obs
0,73935X47,0.486036,0.368292,0.347747,0.663390,33
1,73935X64,0.766246,0.588171,0.501812,0.733627,89
2,73936Q73,0.547452,0.392828,0.424132,0.360545,35
3,73936Q85,0.964516,0.771675,0.722309,1.001637,28
4,74022D40,2.268966,1.844893,1.842990,2.263018,218
...,...,...,...,...,...,...
20546,90467720,1.355817,0.996763,1.022447,1.332434,264
20547,90470L56,0.539628,0.330557,0.337636,0.546592,48
20548,90934C10,0.593219,0.649882,0.825251,0.586968,99
20549,91154910,0.979785,1.138464,1.202114,1.112078,140


In [3]:
conn.close()

NameError: name 'conn' is not defined